In [ ]:
import requests
import import_ipynb
from utils import qrread as qr
import msvcrt
import cv2
import os
from datetime import datetime



SERVER_URL = "http://192.168.1.10:8000/attendance"
DEVICE_ID = "Devf01673"


def scan_user():
    print("\nProgram initiated. \nScanning QR")
    try:
        details = qr.scan_camera()
        print("QR Scanned Sucessfully !!!!")
        print(f"Name: {details["Name"]} \nID : {details["id"]}\n")
    except Exception as e:
        print(f"An error occured : {e} ")
    data = {
        "name": str(details["Name"]),
        "user_id": int(details["id"]),
        "secret_code": str(details["auth_code"]),
        "device_id": str(DEVICE_ID)
    }

    return data


def capture_face(
    model_path=r"files\face_detection_yunet_2026may.onnx"
):
    # Create photos folder
    os.makedirs("photos", exist_ok=True)

    # Open camera
    camera = cv2.VideoCapture(0)

    if not camera.isOpened():
        print("Could not open camera.")
        return None

    # Get camera resolution
    width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Create YuNet face detector
    detector = cv2.FaceDetectorYN_create(
        model_path,
        "",
        (width, height),
        0.6,
        0.3,
        5000
    )

    print("Looking for exactly one face...")
    print("Press Q to quit.")

    while True:
        success, frame = camera.read()

        if not success:
            print("Could not read camera.")
            break

        # Detect faces
        _, faces = detector.detect(frame)

        number_of_faces = 0 if faces is None else len(faces)

        # Exactly one face detected
        if number_of_faces == 1:

            # Date/time filename
            timestamp = datetime.now().strftime(
                "%Y-%m-%d_%H-%M-%S"
            )

            # Save inside photos folder
            filename = os.path.join(
                "photos",
                f"{timestamp}.jpg"
            )

            cv2.imwrite(filename, frame)

            print(f"Photo saved: {filename}")

            camera.release()
            cv2.destroyAllWindows()

            return filename

        # Display status
        cv2.putText(
            frame,
            f"Faces detected: {number_of_faces}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        cv2.imshow("Face Detection", frame)

        # Press Q to quit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    camera.release()
    cv2.destroyAllWindows()

    return None

def send_attendance():
    data = scan_user()
    capture_face()
    try:
        response = requests.post(
            SERVER_URL,
            json=data,
            timeout=5
        )

        print("Status:", response.status_code)
        response.raise_for_status()

        result = response.json()

        if result["success"]:
            print("Attendance Marked Successfully")
        else:
            print("Failed:", result["message"])

        return result

    except requests.exceptions.RequestException as e:
        print("Server Error:", e)
        return None


if __name__ == "__main__":
    while True:
        input("Press Enter key to start.")
        send_attendance()

Press Enter key to start. 



Program initiated. 
Scanning QR
QR Scanned Sucessfully !!!!
Name: Jack1 
ID : 10

Looking for exactly one face...
Press Q to quit.
Photo saved: photos\2026-08-27_15-54-49.jpg
Status: 200
Attendance Marked Successfully


In [8]:
!pip install opencv-contrib-python

   ---------------------------------------- 0.0/53.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/53.8 MB 6.1 MB/s eta 0:00:09
    --------------------------------------- 1.0/53.8 MB 3.0 MB/s eta 0:00:18
   - -------------------------------------- 2.1/53.8 MB 4.1 MB/s eta 0:00:13
   -- ------------------------------------- 2.9/53.8 MB 3.8 MB/s eta 0:00:14
   --- ------------------------------------ 4.2/53.8 MB 4.3 MB/s eta 0:00:12
   --- ------------------------------------ 5.2/53.8 MB 4.5 MB/s eta 0:00:11
   ---- ----------------------------------- 6.6/53.8 MB 4.7 MB/s eta 0:00:10
   ----- ---------------------------------- 7.6/53.8 MB 4.8 MB/s eta 0:00:10
   ------ --------------------------------- 8.9/53.8 MB 5.0 MB/s eta 0:00:10
   ------- -------------------------------- 10.2/53.8 MB 5.1 MB/s eta 0:00:09
   -------- ------------------------------- 11.3/53.8 MB 5.2 MB/s eta 0:00:09
   --------- ------------------------------ 12.6/53.8 MB 5.3 MB/s eta 0:00:08
   

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Kshitij\\anaconda3\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.

